# CRISPRitz Off-Target Search

This notebook demonstrates `run_crispritz_search`, which enumerates every site in a local reference genome that a guide could bind, allowing both mismatches and DNA/RNA bulges. [CRISPRitz](https://github.com/pinellolab/CRISPRitz) builds a PAM-aware index over the reference once, then searches every guide against it.

The search runs locally: it reads a reference FASTA you supply and keeps its index in a local cache, so it does not run on hosted devices.

In [ ]:
from proto_tools.utils.notebook_docs import display_api_reference
from proto_tools.tools.sequence_alignment.crispritz import (
    CRISPRitzSearchConfig,
    CRISPRitzSearchInput,
    run_crispritz_search,
)

## API Reference

In [ ]:
display_api_reference("crispritz-search", "input", "run_crispritz_search")

In [ ]:
display_api_reference("crispritz-search", "config", "run_crispritz_search")

In [ ]:
display_api_reference("crispritz-search", "output", "run_crispritz_search")

## Search a Local Reference

Guides are PAM-free protospacers of a single length; the PAM comes from the config, written in nuclease orientation. `reference_fasta` is an uncompressed FASTA file or a directory of them, such as the genome inside an `ncbi-reference-bundle` result.

In [ ]:
from pathlib import Path

reference = Path("../example_reference.fasta")
result = run_crispritz_search(
    CRISPRitzSearchInput(
        guides=["GACTGCATTGCATGACTGCA"],
        reference_fasta=str(reference),
        reference_id="synthetic-crispritz-example-v1",
    ),
    CRISPRitzSearchConfig(max_mismatches=4, max_dna_bulges=1, max_rna_bulges=1),
)

print(f"{len(result.hits)} hits, index reused: {result.index_reused}")
for hit in result.hits:
    print(f"{hit.chromosome}:{hit.start}-{hit.end} {hit.strand} {hit.bulge_type} mm={hit.mismatches}")

## Read an Alignment

`guide_alignment` and `target_alignment` are the aligned strings CRISPRitz produced, with `-` marking a bulge, so a hit can be inspected base by base. Coordinates are 1-indexed and inclusive, like everywhere else in Proto.

In [ ]:
for hit in result.hits[:3]:
    print(f"guide  {hit.guide_alignment}")
    print(f"target {hit.target_alignment}")
    print(f"       {hit.mismatches} mismatches, {hit.bulge_size}-nt {hit.bulge_type} bulge\n")

## Tighten the Search

Mismatch and bulge limits are inclusive maxima, and `max_total_edits` caps mismatches plus bulge length together, which is usually the cheaper way to cut a large hit list down.

In [ ]:
strict = run_crispritz_search(
    CRISPRitzSearchInput(guides=["GACTGCATTGCATGACTGCA"], reference_fasta=str(reference)),
    CRISPRitzSearchConfig(max_mismatches=2, max_dna_bulges=0, max_rna_bulges=0, max_total_edits=2),
)
print(f"{len(result.hits)} hits with the permissive settings, {len(strict.hits)} with the strict ones")

## Export Results

JSON keeps the search parameters and reference checksum alongside the hits; CSV flattens to one row per hit.

In [ ]:
output_dir = Path("./example_output")
output_dir.mkdir(exist_ok=True)

result.export(name="crispritz_hits", export_path=output_dir, file_format="csv")
print(f"Off-target hits exported to {output_dir}")